In [1]:
import sqlite3
import pandas as pd
import numpy as np

In [2]:
conn = sqlite3.connect("order_management_0.sqlite")
cursor = conn.cursor()

def col_names(table_name):

    cursor.execute(f"PRAGMA table_info({table_name});")
    columns_info = cursor.fetchall()
    column_names = [column[1] for column in columns_info]

    return column_names

In [3]:
def col_names(table_name, cursor):

    cursor.execute(f"PRAGMA table_info({table_name});")
    columns_info = cursor.fetchall()
    column_names = [column[1] for column in columns_info]

    return column_names


In [4]:
def generate_trace(viewpoint_obj):    
    table = "object_object"
    cols = col_names(table, cursor)
    cursor.execute(f"SELECT * FROM {table} WHERE {cols[0]} = '{viewpoint_obj}'")
    table = cursor.fetchall()
    Set_corr_obj = tuple([row[1] for row in table])
        
    table = "event_object"
    cols = col_names(table, cursor)
    if len(Set_corr_obj) > 1:
        cursor.execute(f"SELECT {cols[0]} FROM {table} WHERE {cols[1]} IN {Set_corr_obj}")
    else:
        cursor.execute(f"SELECT {cols[0]} FROM {table} WHERE {cols[1]} = '{Set_corr_obj[0]}'")
    Set_events = tuple([a[0] for a in set(cursor.fetchall())])
    
    
    
    
    table = "event_object"
    cols = col_names(table, cursor)
    cursor.execute(f"SELECT {cols[1]} FROM {table} WHERE {cols[0]} IN {Set_events}")
    temp_tab = cursor.fetchall()
    Set_obj = tuple(set([a[0] for a in temp_tab]))
    
    table_1 = "object"
    table_2 = "object_map_type"
    cols_1 = col_names(table_1, cursor)
    cols_2 = col_names(table_2, cursor)
        
    cursor.execute(f"SELECT {table_1}.{cols_1[0]}, {table_2}.{cols_2[1]} FROM {table_1}\
                    LEFT JOIN {table_2} ON {table_1}.{cols_1[1]} = {table_2}.{cols_2[0]}\
                    WHERE {table_1}.{cols_1[0]} IN {Set_obj}")
    
    Set_obj = cursor.fetchall()    
    
    table_1 = "event"
    table_2 = "event_map_type"
    cols_1 = col_names(table_1, cursor)
    cols_2 = col_names(table_2, cursor)
    
    cursor.execute(f"SELECT {table_1}.{cols_1[0]}, {table_2}.{cols_2[1]} FROM {table_1}\
                    LEFT JOIN {table_2} ON {table_1}.{cols_1[1]} = {table_2}.{cols_2[0]}\
                    WHERE {table_1}.{cols_1[0]} IN {Set_events}")
    tab = cursor.fetchall()
    
    for idx,row in enumerate(tab):
        temp = "event_" + row[1]
        cursor.execute(f"SELECT * FROM {temp} WHERE {col_names(temp,cursor)[0]} = '{row[0]}'")
        table = cursor.fetchall()
        tab[idx] += (table[0][1], int(viewpoint_obj[-4:]))
        
    
    tab = sorted(tab, key=lambda x: x[2])
    
    return tab

In [5]:
table = "object_Orders"
cols = col_names(table, cursor)
cursor.execute(f"SELECT {cols[0]} FROM {table}")
all_orders = [a[0] for a in cursor.fetchall()]

In [6]:
pd_df = []
order_idx = []
for idx,order in enumerate(all_orders):
    if idx % 100 == 0:
        print(idx)
    temp = generate_trace(order)
    pd_df.extend(temp)
    order_idx.extend([idx + 1] * len(temp))

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900


In [7]:
pd_df = pd.DataFrame(pd_df)
pd_df[3] = order_idx
pd_df.to_csv('order_management_0.csv', index = False)